In [0]:
from datetime import date, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
%run ../../../utils/reader_utils

In [0]:
%run ../../../utils/writer_utils

In [0]:
# Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())
today = date.today()

ENV = configs.get("env", "dev")
INITIAL_RUN = configs.get('initial_run', False)
START_DATE = configs.get("start_date") or today - timedelta(days=1)
END_DATE = configs.get("end_date") or today + timedelta(days=1)
CATALOG = f"sl_{ENV}"

print(ENV, INITIAL_RUN, START_DATE, END_DATE, CATALOG)

In [0]:
# Constants
SOURCE_TABLE_CONF = {
    "table": "vehicle_telemetry_raw",
    "schema": "bronze",
    "dedupe_asc": ["reading_id"],
    "order_col": "reading_timestamp",
    "timestamp_col": "_ingested_at"
}

TARGET_TABLE_CONF = {
    "table": "pyspark_vehicle_telemetry",
    "schema": "silver",
    "merge_keys": ["reading_id"],
    "primary_key": "reading_id"
}

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
vehicle_telemetry_df = read_delta_table(SOURCE_TABLE_CONF, START_DATE, END_DATE)

In [0]:
display(vehicle_telemetry_df)

In [0]:
normalized_vehicle_telemetry_df = (vehicle_telemetry_df.filter(F.col("reading_id").isNotNull())
        .withColumn('reading_timestamp', F.to_timestamp(F.col("reading_timestamp")))
            .withColumn('cargo_temp_c', F.col("cargo_temp_c").cast(DoubleType()))
            .withColumn('cargo_temp_c', 
                        F.when((F.col("cargo_temp_c") < -273) | (F.col("cargo_temp_c") > 100), F.lit(None))
                        .otherwise(F.col("cargo_temp_c")))
            .withColumn('engine_temp_c', F.col("engine_temp_c").cast(DoubleType()))
            .withColumn('engine_temp_c', 
                        F.when((F.col("engine_temp_c") < 0) | (F.col("engine_temp_c") > 200), F.lit(None))
                        .otherwise(F.col("engine_temp_c")))
            .withColumn('fuel_pct', F.col("fuel_pct").cast(DoubleType()))
            .withColumn('odometer_km', F.col("odometer_km").cast(IntegerType()))
            .withColumn('speed_kmh', F.col("speed_kmh").cast(DoubleType()))
            .withColumn('speed_kmh', 
                        F.when((F.col("speed_kmh") < 0) | (F.col("speed_kmh") > 180), F.lit(None))
                        .otherwise(F.col("speed_kmh")))
            .select("cargo_temp_c",
                        "engine_temp_c",
                        "fuel_pct",
                        "odometer_km",
                        "reading_id",
                        "reading_timestamp",
                        "speed_kmh",
                        "vehicle_id",
                        "vehicle_status"))

In [0]:
enriched_vehicle_telemetry_df = (normalized_vehicle_telemetry_df
                                 .withColumn('is_cold_chain_cargo', F.when(F.col("cargo_temp_c").isNotNull(), True).otherwise(F.lit(False)))
            .withColumn('speed_limit_exceed', F.when(F.col("speed_kmh") > 120, True).otherwise(False))
            .withColumn('engine_overheat', F.when(F.col("engine_temp_c") > 110, True).otherwise(False))
            .withColumn('reading_date', F.to_date(F.col("reading_timestamp")))
            )

In [0]:
if INITIAL_RUN: #this is not an ideal production pattern but more straight forward for this used case
    target_table = f"{TARGET_TABLE_CONF.get('schema')}.{TARGET_TABLE_CONF.get('table')}"
    primary_key = TARGET_TABLE_CONF.get('primary_key')

    (enriched_vehicle_telemetry_df
     .writeTo(target_table)
     .using("delta")
     .createOrReplace())
    
    spark.sql(f"ALTER TABLE {target_table} CLUSTER BY (reading_date, vehicle_id, vehicle_status)")
    spark.sql(f"ALTER TABLE {target_table} ALTER COLUMN {primary_key} SET NOT NULL")
    spark.sql(f"ALTER TABLE {target_table} ADD CONSTRAINT {primary_key}_pk PRIMARY KEY ({primary_key})")
    
    dbutils.notebook.exit(f"Initial Table: {target_table} Setup Finished Successfully")

In [0]:
upsert_to_delta(enriched_vehicle_telemetry_df, TARGET_TABLE_CONF)